# OSINT Analytical Platform - Complete Pipeline

Este notebook demuestra el pipeline completo de:
1. Embeddings semánticos
2. Topic modeling avanzado (BERTopic)
3. Búsqueda semántica
4. Modelos ML avanzados

**Nota**: Para usar este notebook en Jupyter, ejecute las celdas en orden.

## 1. Setup e Imports

In [1]:
from pathlib import Path
import sys

sys.path.append(str(Path('scripts').resolve()))

import pandas as pd
import numpy as np
import pickle
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

print('✅ Imports successful')

✅ Imports successful


## 2. Cargar Dataset Base

In [3]:
df = pd.read_csv('data/dataset_nlp_labeled.csv', parse_dates=['timestamp'])
print(f'📊 Dataset loaded: {len(df)} documents')
print(f'\nColumns: {df.columns.tolist()}')
print(f'\nFirst 3 documents:')
df[['timestamp', 'source', 'title', 'weak_label']].head(3)

FileNotFoundError: [Errno 2] No such file or directory: 'data/dataset_nlp_labeled.csv'

## 3. Semantic Embeddings

In [ ]:
# Load embeddings
with open('data/dataset_nlp_embeddings.pkl', 'rb') as f:
    embeddings_data = pickle.load(f)

embeddings = embeddings_data['embeddings']
umap_embeddings = embeddings_data['umap']

print(f'📌 Embeddings loaded:')
print(f'  - Shape: {embeddings.shape}')
print(f'  - UMAP: {umap_embeddings.shape}')
print(f'  - Model: all-MiniLM-L6-v2 (384 dimensions)')

### 3.1 Visualize Semantic Space

In [ ]:
# Show UMAP visualization
img_path = Path('outputs/advanced_figures/semantic_space_umap.png')
if img_path.exists():
    img = Image.open(img_path)
    plt.figure(figsize=(12, 8))
    plt.imshow(img)
    plt.axis('off')
    plt.title('Semantic Space Visualization (UMAP)', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ UMAP visualization not found')

## 4. Topic Modeling with BERTopic

In [ ]:
# Load BERTopic results
bertopic_df = pd.read_csv('data/dataset_nlp_bertopic.csv', parse_dates=['timestamp'])
topics_df = pd.read_csv('outputs/models/bertopic_topics.csv')

print(f'🎯 BERTopic Results:')
print(f'  - Unique topics: {len(topics_df)}')
print(f'  - Documents analyzed: {len(bertopic_df)}')
print(f'\nTop 5 topics:')
topics_df.head(5)[['Topic', 'Count', 'Name']].to_string(index=False)

In [ ]:
# Topic distribution
topic_counts = bertopic_df['bertopic_id'].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(12, 6))
topic_counts.plot(kind='bar', ax=ax, color='steelblue', edgecolor='black')
ax.set_xlabel('Topic ID', fontsize=11, fontweight='bold')
ax.set_ylabel('Document Count', fontsize=11, fontweight='bold')
ax.set_title('Document Distribution by BERTopic', fontsize=13, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Topic distribution: {topic_counts.to_dict()}')

## 5. Semantic Search Examples

In [ ]:
# Load search engine
from scripts.semantic_search import SemanticSearchEngine

engine = SemanticSearchEngine(embeddings, df)

# Example searches
queries = [
    'Israel Iran military conflict',
    'diplomatic negotiations ceasefire',
    'sanctions embargo energy',
]

for query in queries:
    print(f'\n🔍 Query: "{query}"')
    print('─' * 80)
    results = engine.search(query, k=3)
    for result in results:
        print(f"  [{result['rank']}] {result['similarity']:.3f} | {result['title'][:70]}")
        print(f"       Source: {result['source']} | Label: {result['weak_label']}")

## 6. Advanced ML Models

In [ ]:
# Load model comparison results
ml_results = pd.read_csv('outputs/model_comparison_advanced.csv')

print('🤖 ML Models Performance:')
print(ml_results.to_string(index=False))
print(f'\n✅ Best Model: {ml_results.loc[ml_results["accuracy"].idxmax(), "model"]}')
print(f'   Accuracy: {ml_results["accuracy"].max():.4f}')
print(f'   F1 Macro: {ml_results.loc[ml_results["accuracy"].idxmax(), "f1_macro"]:.4f}')

### 6.1 Model Comparison Visualization

In [ ]:
img_path = Path('outputs/advanced_figures/model_comparison_advanced.png')
if img_path.exists():
    img = Image.open(img_path)
    plt.figure(figsize=(14, 5))
    plt.imshow(img)
    plt.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print('⚠️ Model comparison visualization not found')

### 6.2 Confusion Matrices

In [ ]:
# Display confusion matrices for top models
top_models = ['GradientBoosting', 'RandomForest', 'LogisticRegression']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for idx, model_name in enumerate(top_models):
    cm_path = Path(f'outputs/advanced_figures/confusion_matrix_{model_name.lower()}.png')
    if cm_path.exists():
        img = Image.open(cm_path)
        axes[idx].imshow(img)
        axes[idx].axis('off')
        axes[idx].set_title(model_name, fontweight='bold')

plt.tight_layout()
plt.show()

## 7. Summary Statistics

In [ ]:
print('📈 Project Summary')
print('═' * 70)
print(f'\n📊 Dataset:')
print(f'  - Total documents: {len(df)}')
print(f'  - Date range: {df["timestamp"].min().date()} to {df["timestamp"].max().date()}')
print(f'  - Sources: {df["source"].nunique()} ({df["source"].unique().tolist()})')
print(f'  - Avg text length: {df["processed_text"].str.len().mean():.0f} chars')

print(f'\n🏷️ Labels:')
for label, count in df['weak_label'].value_counts().items():
    pct = 100 * count / len(df)
    print(f'  - {label}: {count} ({pct:.1f}%)')

print(f'\n🎯 BERTopic:')
print(f'  - Topics detected: {len(topics_df)}')
print(f'  - Largest topic: {topic_counts.max()} documents')
print(f'  - Smallest topic: {topic_counts.min()} documents')

print(f'\n🤖 ML Models:')
print(f'  - Best accuracy: {ml_results["accuracy"].max():.4f}')
print(f'  - Best model: {ml_results.loc[ml_results["accuracy"].idxmax(), "model"]}')
print(f'  - Best F1 macro: {ml_results["f1_macro"].max():.4f}')

print(f'\n📌 Embeddings:')
print(f'  - Dimension: {embeddings.shape[1]}')
print(f'  - Model: all-MiniLM-L6-v2')
print(f'  - UMAP reduced to: {umap_embeddings.shape[1]}D')

print(f'\n✅ Platform Ready for Interactive Dashboard')
print(f'   Run: streamlit run app.py')